# Data Quality Summary

## Dataset Overview
* **Total Records:** 541,910
* **Total Columns:** 8
* **Time Period:** December 1, 2010 - December 9, 2011
* **Primary Objective:** Address critical anomalies identified in customer identifiers, transactional amounts, and product descriptions prior to performing exploratory and financial analysis.

---

## Key Findings & Action Plan

### 1. Missing Values 
* **Customer ID :** Approximately 25% of the dataset (135,080 rows) lacks a Customer ID. 
  > **Business Action:** As these transactions cannot be attributed to specific users for RFM and cohort analysis, they are excluded from the customer-centric modeling layer.
* **Description:** 1,454 rows contain no product description. 
  > **Business Action:** Resolved by imputing missing values with the placeholder 'Unknown Item'.

### 2. Invalid & Extreme Values 
* **Negative Quantities & Cancellations:** 10,624 transactions with exhibit quantities <= 0. Among these, 9,288 represent official cancellations (invoices prefixed with 'C'). Extreme values ranging from -80,995 to 80,995 indicate significant inventory adjustments.
* **Pricing Anomalies:** 
  * Minimum Price: -11,062.06 (indicative of a system error or financial adjustment).
  * Maximum Price: 38,970.00 (a severe outlier compared to the median price of 2.08).
* > **Business Action:** All transactions involving cancellations, zero, or negative prices and quantities are filtered out to ensure financial metrics accurately reflect actual revenue.

### 3. Data Duplication 
* **Redundant Records:** The dataset contains 5,268 exact duplicate rows, likely resulting from repetitive system logging.
  > **Business Action:** Duplicates are dropped entirely to prevent the artificial inflation of sales volumes and transaction counts.

### 4. Geographical Distribution & Textual Inconsistency 
* **Market Imbalance:** The United Kingdom dominates the dataset, accounting for ~91.4% (495,478 rows) of all records. The second-largest market, Germany, contains only 9,495 rows.
* **Naming Inconsistencies:** Ireland is recorded under two distinct names: "EIRE" (8,196 rows) and "Ireland".
  > **Business Action:** Standardize "EIRE" to "Ireland" to ensure accurate geographical aggregation and reporting.

In [42]:
import pandas as pd
df = pd.read_excel('../data/online_retail_II.xlsx', sheet_name='Year 2010-2011')

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      541910 non-null  object        
 1   StockCode    541910 non-null  object        
 2   Description  540456 non-null  object        
 3   Quantity     541910 non-null  int64         
 4   InvoiceDate  541910 non-null  datetime64[ns]
 5   Price        541910 non-null  float64       
 6   Customer ID  406830 non-null  float64       
 7   Country      541910 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB
None


In [43]:
print(df.describe())


            Quantity                    InvoiceDate          Price  \
count  541910.000000                         541910  541910.000000   
mean        9.552234  2011-07-04 13:35:22.342307584       4.611138   
min    -80995.000000            2010-12-01 08:26:00  -11062.060000   
25%         1.000000            2011-03-28 11:34:00       1.250000   
50%         3.000000            2011-07-19 17:17:00       2.080000   
75%        10.000000            2011-10-19 11:27:00       4.130000   
max     80995.000000            2011-12-09 12:50:00   38970.000000   
std       218.080957                            NaN      96.759765   

         Customer ID  
count  406830.000000  
mean    15287.684160  
min     12346.000000  
25%     13953.000000  
50%     15152.000000  
75%     16791.000000  
max     18287.000000  
std      1713.603074  


In [44]:
print(df.isnull().sum())

Invoice             0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
Price               0
Customer ID    135080
Country             0
dtype: int64


In [45]:
print(df['Invoice'].str.startswith('C').sum())   
print((df['Quantity'] <= 0).sum())               

9288
10624


In [46]:
df['Description'] = df['Description'].fillna('Unknown Item')
print(df['Description'].isnull().sum())


0


In [47]:

#df_clean = df.dropna(subset=['Customer ID'])


In [48]:
print(f"{df.shape[0]}")

541910


In [49]:
df['Customer ID'] = df['Customer ID'].astype('Int64')


In [50]:
print(df['Customer ID'].head())
print("---------------------------------")
print(df.info())

0    17850
1    17850
2    17850
3    17850
4    17850
Name: Customer ID, dtype: Int64
---------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      541910 non-null  object        
 1   StockCode    541910 non-null  object        
 2   Description  541910 non-null  object        
 3   Quantity     541910 non-null  int64         
 4   InvoiceDate  541910 non-null  datetime64[ns]
 5   Price        541910 non-null  float64       
 6   Customer ID  406830 non-null  Int64         
 7   Country      541910 non-null  object        
dtypes: Int64(1), datetime64[ns](1), float64(1), int64(1), object(4)
memory usage: 33.6+ MB
None


In [51]:
print(df['Country'].value_counts().head(10))


Country
United Kingdom    495478
Germany             9495
France              8558
EIRE                8196
Spain               2533
Netherlands         2371
Belgium             2069
Switzerland         2002
Portugal            1519
Australia           1259
Name: count, dtype: int64


In [52]:
print(df[df['Country'].str.contains('ire', case=False, na=False)]['Country'].unique())

['EIRE']


In [53]:
df['Country'] = df['Country'].replace('EIRE', 'Ireland')

analysing saparate - UK and other countries 

In [54]:
print(df.duplicated().sum())

5268


In [55]:
%store df

Stored 'df' (DataFrame)
